# 1. Experiment Overview

- 한국 판례 Hugging Face dataset 로드
- 미국 판례 CourtListener API 수집
- 한국/미국 판례 공통 schema normalize
- damages/tort 관련 판례 필터링
- 최소 길이 필터링
- jurisdiction별 파일럿 샘플링
- cleaned CSV 저장


# 2. Environment Setup

- 권장 Python 버전: `3.11.9`
- 로컬 VSCode + Jupyter 환경 기준
- 로컬 상대경로 사용: `./data`, `./outputs`, `./outputs/figures`
- 한국 판례: Hugging Face `datasets`
- 미국 판례: CourtListener REST API


# 3. Imports and Global Config

In [ ]:
from __future__ import annotations

import random
import time
import warnings
from html import unescape
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import requests
from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm
import nltk
import re

try:
    import regex as regex_module
except ImportError:
    regex_module = re

try:
    import kss
    KSS_AVAILABLE = True
except ImportError:
    kss = None
    KSS_AVAILABLE = False

for resource_name in ["punkt", "punkt_tab"]:
    try:
        nltk.download(resource_name, quiet=True)
    except Exception as exc:
        print(f"[warning] nltk resource download failed for {resource_name}: {exc}")

pd.set_option("display.max_colwidth", 160)
warnings.simplefilter("always", UserWarning)
tqdm.pandas()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

KR_DATASET_NAME = "lbox/lbox_open"
KR_DATASET_CONFIG_NAME = "precedent_corpus"
KR_SPLIT = "train"

KR_COLUMN_MAP = {
    "case_id": "id",
    "court_name": None,
    "court_level": None,
    "decision_date": None,
    "case_name": None,
    "case_number_or_citation": None,
    "source": None,
    "case_type_keyword": None,
    "raw_text": "precedent",
}

COURTLISTENER_API_TOKEN = "b2c082d811265309d8f0bfa769caedfb77499cb5"
COURTLISTENER_SEARCH_API_URL = "https://www.courtlistener.com/api/rest/v4/search/"
COURTLISTENER_OPINIONS_API_URL = "https://www.courtlistener.com/api/rest/v4/opinions/"
COURTLISTENER_TIMEOUT_SEC = 60
COURTLISTENER_REQUEST_SLEEP_SEC = 13.0
COURTLISTENER_RATE_LIMIT_PER_MINUTE = 5
COURTLISTENER_RATE_LIMIT_PER_HOUR = 50
COURTLISTENER_RATE_LIMIT_PER_DAY = 125
COURTLISTENER_CALLS_USED_THIS_MINUTE = 0
COURTLISTENER_CALLS_USED_THIS_HOUR = 0
COURTLISTENER_CALLS_USED_TODAY = 50

US_USE_SEMANTIC_SEARCH = False
US_SEARCH_TYPE = "o"
US_MAX_SEARCH_RESULTS = 50
US_SEARCH_QUERY = '("damages" OR "civil damages" OR negligence OR tort OR "duty of care" OR "proximate cause" OR "comparative negligence" OR "compensatory damages" OR "punitive damages" OR "emotional distress" OR "personal injury" OR "breach of duty")'
US_EXTRA_QUERY = ""
US_SEARCH_PARAMS = {
    "type": US_SEARCH_TYPE,
}
US_PARTIAL_SAVE_EVERY = 5

SAMPLE_SIZE_PER_JURISDICTION = 50
MIN_TOKENS = 800
REMOVE_KR_APPENDIX_AFTER_BYEOLJI = True
PREVIEW_ROWS = 3

KR_KEYWORDS = [
    "손해배상",
    "손해배상(기)",
    "손해배상(의)",
    "불법행위",
    "위자료",
    "지연손해금",
    "과실상계",
    "상당인과관계",
    "민법 제750조",
    "민법 제393조",
    "민법 제763조",
]

US_KEYWORDS = [
    "damages",
    "civil damages",
    "negligence",
    "tort",
    "duty of care",
    "proximate cause",
    "comparative negligence",
    "compensatory damages",
    "punitive damages",
    "emotional distress",
    "personal injury",
    "breach of duty",
]

COMMON_SCHEMA_COLUMNS = [
    "case_id",
    "jurisdiction",
    "country",
    "court_name",
    "court_level",
    "decision_date",
    "case_name",
    "case_number_or_citation",
    "source",
    "case_type_keyword",
    "raw_text",
    "clean_text",
    "reasoning_text",
    "token_count",
    "sentence_count",
]

ESTIMATED_SEARCH_PAGE_SIZE = 20

def estimate_courtlistener_request_count(max_results: int, page_size: int = ESTIMATED_SEARCH_PAGE_SIZE) -> int:
    if max_results <= 0:
        return 0
    return max_results + int(np.ceil(max_results / page_size))

estimated_request_count = estimate_courtlistener_request_count(US_MAX_SEARCH_RESULTS)
estimated_minutes = estimated_request_count / 5
remaining_daily_budget = COURTLISTENER_RATE_LIMIT_PER_DAY - COURTLISTENER_CALLS_USED_TODAY
remaining_hourly_budget = COURTLISTENER_RATE_LIMIT_PER_HOUR - COURTLISTENER_CALLS_USED_THIS_HOUR
remaining_minute_budget = COURTLISTENER_RATE_LIMIT_PER_MINUTE - COURTLISTENER_CALLS_USED_THIS_MINUTE

print(f"BASE_DIR: {BASE_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"KSS_AVAILABLE: {KSS_AVAILABLE}")
print(f"US_USE_SEMANTIC_SEARCH: {US_USE_SEMANTIC_SEARCH}")
print(f"Estimated US API requests: ~{estimated_request_count}")
print(f"Estimated minimum runtime at 5 req/min: ~{estimated_minutes:.1f} minutes")
print(f"Remaining CourtListener daily budget: {remaining_daily_budget}")
print(f"Remaining CourtListener hourly budget: {remaining_hourly_budget}")
print(f"Remaining CourtListener minute budget: {remaining_minute_budget}")


BASE_DIR: C:\Users\이정연\OneDrive\바탕 화면\comparative-law-llm\comparative-law-llm
OUTPUT_DIR: C:\Users\이정연\OneDrive\바탕 화면\comparative-law-llm\comparative-law-llm\outputs
KSS_AVAILABLE: True
US_USE_SEMANTIC_SEARCH: False
Estimated US API requests: ~53
Estimated minimum runtime at 5 req/min: ~10.6 minutes
Remaining CourtListener daily budget: 77
Remaining CourtListener hourly budget: 50
Remaining CourtListener minute budget: 5


# 4. Load Korean Dataset from Hugging Face and Collect U.S. Cases from CourtListener API

미국 판례는 CourtListener `search` API로 후보 opinion cluster를 찾고, 각 결과의 nested opinion ID를 이용해 `opinions` API에서 full opinion text를 가져옵니다. 재현성과 제어 가능성을 위해 기본값은 keyword search이며, `US_USE_SEMANTIC_SEARCH = True`로 바꾸면 `semantic=true` 파라미터를 추가할 수 있습니다.


In [2]:
def is_missing_value(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, (list, tuple, set, dict, np.ndarray, pd.Series)):
        return False
    try:
        missing = pd.isna(value)
    except (TypeError, ValueError):
        return False
    return bool(missing) if isinstance(missing, (bool, np.bool_)) else False


def stringify_value(value: Any) -> Optional[str]:
    if is_missing_value(value):
        return None
    if isinstance(value, dict):
        parts = []
        for key, item in value.items():
            item_text = stringify_value(item)
            if item_text:
                parts.append(f"{key}: {item_text}")
        return "\n".join(parts).strip() or None
    if isinstance(value, (list, tuple, set)):
        parts = [item for item in (stringify_value(v) for v in value) if item]
        return "\n".join(parts).strip() or None
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return None
    return text


def display_preview(df: pd.DataFrame, label: str, rows: int = PREVIEW_ROWS) -> None:
    print(f"{label} shape: {df.shape}")
    if df.empty:
        print(f"{label} is empty.")
        return
    display(df.head(rows))


class CourtListenerStop(RuntimeError):
    pass


class CourtListenerRateLimitReached(CourtListenerStop):
    pass


class CourtListenerBudgetExceeded(CourtListenerStop):
    pass


class CourtListenerSearchStopped(CourtListenerStop):
    def __init__(self, message: str, collected_results: list[dict[str, Any]]):
        super().__init__(message)
        self.collected_results = list(collected_results)


def validate_kr_dataset_config(dataset_name: str, config_name: str, split_name: str, column_map: dict[str, Optional[str]]) -> None:
    if not dataset_name or dataset_name.startswith("TODO"):
        raise ValueError("KR_DATASET_NAME must be updated before running this notebook.")
    if not config_name or config_name.startswith("TODO"):
        raise ValueError("KR_DATASET_CONFIG_NAME must be updated before running this notebook.")
    if not split_name:
        raise ValueError("KR_SPLIT must be a non-empty string.")
    raw_text_column = column_map.get("raw_text")
    if raw_text_column is None or raw_text_column == "TODO":
        raise ValueError("KR_COLUMN_MAP['raw_text'] must point to an actual dataset column.")


def validate_us_api_config(api_token: str, search_url: str, opinions_url: str) -> None:
    if not api_token or api_token.startswith("TODO"):
        raise ValueError("COURTLISTENER_API_TOKEN must be updated before running this notebook.")
    if not search_url.startswith("https://"):
        raise ValueError("COURTLISTENER_SEARCH_API_URL must be an https URL.")
    if not opinions_url.startswith("https://"):
        raise ValueError("COURTLISTENER_OPINIONS_API_URL must be an https URL.")
    if COURTLISTENER_CALLS_USED_THIS_MINUTE >= COURTLISTENER_RATE_LIMIT_PER_MINUTE:
        raise ValueError("COURTLISTENER_CALLS_USED_THIS_MINUTE already meets/exceeds the per-minute limit.")
    if COURTLISTENER_CALLS_USED_THIS_HOUR >= COURTLISTENER_RATE_LIMIT_PER_HOUR:
        raise ValueError("COURTLISTENER_CALLS_USED_THIS_HOUR already meets/exceeds the per-hour limit.")
    if COURTLISTENER_CALLS_USED_TODAY >= COURTLISTENER_RATE_LIMIT_PER_DAY:
        raise ValueError("COURTLISTENER_CALLS_USED_TODAY already meets/exceeds the per-day limit.")


def load_hf_dataset_as_dataframe(dataset_name: str, config_name: Optional[str], split_name: str, label: str) -> pd.DataFrame:
    config_msg = f" | config={config_name}" if config_name else ""
    print(f"Loading {label} dataset: {dataset_name}{config_msg} | split={split_name}")
    if config_name:
        dataset = load_dataset(dataset_name, config_name, split=split_name)
    else:
        dataset = load_dataset(dataset_name, split=split_name)
    dataframe = dataset.to_pandas()
    print(f"{label} columns ({len(dataframe.columns)}): {list(dataframe.columns)}")
    display_preview(dataframe, f"{label} raw dataset")
    return dataframe


def build_courtlistener_headers(api_token: str) -> dict[str, str]:
    return {
        "Authorization": f"Token {api_token}",
        "Accept": "application/json",
        "User-Agent": "comparative-law-llm/1.0",
    }


def initialize_courtlistener_request_state() -> dict[str, int]:
    return {
        "made_this_run": 0,
        "minute_used_at_start": COURTLISTENER_CALLS_USED_THIS_MINUTE,
        "hour_used_at_start": COURTLISTENER_CALLS_USED_THIS_HOUR,
        "day_used_at_start": COURTLISTENER_CALLS_USED_TODAY,
    }


def assert_courtlistener_request_budget(request_state: dict[str, int]) -> None:
    next_hour_total = request_state["hour_used_at_start"] + request_state["made_this_run"] + 1
    next_day_total = request_state["day_used_at_start"] + request_state["made_this_run"] + 1
    if next_hour_total > COURTLISTENER_RATE_LIMIT_PER_HOUR:
        raise CourtListenerBudgetExceeded("Per-hour CourtListener budget would be exceeded by the next request.")
    if next_day_total > COURTLISTENER_RATE_LIMIT_PER_DAY:
        raise CourtListenerBudgetExceeded("Per-day CourtListener budget would be exceeded by the next request.")


def build_us_search_query(base_query: str, extra_query: str = "") -> str:
    base_query = base_query.strip()
    extra_query = extra_query.strip()
    if not extra_query:
        return base_query
    return f"({base_query}) AND ({extra_query})"


def courtlistener_get_json(
    url: str,
    headers: dict[str, str],
    request_state: dict[str, int],
    params: Optional[dict[str, Any]] = None,
    timeout_sec: int = COURTLISTENER_TIMEOUT_SEC,
    sleep_sec: float = COURTLISTENER_REQUEST_SLEEP_SEC,
) -> dict[str, Any]:
    assert_courtlistener_request_budget(request_state)
    response = requests.get(url, headers=headers, params=params, timeout=timeout_sec)
    request_state["made_this_run"] += 1
    if response.status_code == 429:
        raise CourtListenerRateLimitReached("CourtListener API rate limit reached.")
    response.raise_for_status()
    payload = response.json()
    if sleep_sec > 0:
        time.sleep(sleep_sec)
    return payload


def fetch_courtlistener_search_results(
    search_url: str,
    api_token: str,
    query: str,
    max_results: int,
    use_semantic_search: bool,
    request_state: dict[str, int],
    extra_params: Optional[dict[str, Any]] = None,
) -> list[dict[str, Any]]:
    headers = build_courtlistener_headers(api_token)
    params = dict(extra_params or {})
    params["q"] = query
    if use_semantic_search:
        params["semantic"] = "true"

    collected: list[dict[str, Any]] = []
    seen_cluster_ids: set[Any] = set()
    next_url: Optional[str] = search_url
    next_params: Optional[dict[str, Any]] = params
    page_idx = 0

    while next_url and len(collected) < max_results:
        try:
            payload = courtlistener_get_json(next_url, headers=headers, request_state=request_state, params=next_params)
        except CourtListenerStop as exc:
            raise CourtListenerSearchStopped(str(exc), collected) from exc
        page_results = payload.get("results", [])
        if not page_results:
            break

        for result in page_results:
            cluster_id = result.get("cluster_id")
            if cluster_id in seen_cluster_ids:
                continue
            seen_cluster_ids.add(cluster_id)
            collected.append(result)
            if len(collected) >= max_results:
                break

        page_idx += 1
        print(f"CourtListener search page {page_idx}: collected {len(collected)} results")
        next_url = payload.get("next")
        next_params = None

    return collected[:max_results]


def select_preferred_opinion_meta(opinions: Any) -> Optional[dict[str, Any]]:
    if not isinstance(opinions, list) or not opinions:
        return None
    priority = {
        "lead-opinion": 0,
        "combined-opinion": 1,
        "plurality-opinion": 2,
        "on-the-merits": 3,
    }
    opinion_candidates = [item for item in opinions if isinstance(item, dict) and item.get("id") is not None]
    if not opinion_candidates:
        return None
    opinion_candidates.sort(key=lambda item: priority.get(str(item.get("type", "")).lower(), 99))
    return opinion_candidates[0]


def fetch_opinion_detail(opinions_api_url: str, api_token: str, opinion_id: Any, request_state: dict[str, int]) -> dict[str, Any]:
    opinion_url = f"{opinions_api_url.rstrip('/')}/{opinion_id}/"
    headers = build_courtlistener_headers(api_token)
    return courtlistener_get_json(opinion_url, headers=headers, request_state=request_state)


def strip_html_like_markup(text: str) -> str:
    text = re.sub(r"<script.*?>.*?</script>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<style.*?>.*?</style>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<[^>]+>", " ", text)
    text = unescape(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_us_raw_text_from_opinion_payload(opinion_payload: dict[str, Any]) -> Optional[str]:
    candidate_fields = [
        "html_with_citations",
        "plain_text",
        "html",
        "html_lawbox",
        "html_columbia",
        "html_anon_2020",
        "xml_harvard",
    ]
    for field_name in candidate_fields:
        value = stringify_value(opinion_payload.get(field_name))
        if value:
            if field_name.startswith("html") or field_name.startswith("xml"):
                return strip_html_like_markup(value)
            return value
    return None


def infer_us_court_level(court_name: Optional[str]) -> str:
    name = (court_name or "").lower()
    if "supreme court" in name:
        return "supreme"
    if "court of appeals" in name or "appellate" in name:
        return "appellate"
    if "district court" in name or "superior court" in name or "trial" in name:
        return "trial"
    return "unknown"


def build_us_cases_dataframe(rows: list[dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(rows, columns=COMMON_SCHEMA_COLUMNS)


def save_us_partial_cases(rows: list[dict[str, Any]], suffix: str = "partial") -> Path:
    partial_path = OUTPUT_DIR / f"us_cases_raw_{suffix}.csv"
    partial_df = build_us_cases_dataframe(rows)
    partial_df.to_csv(partial_path, index=False, encoding="utf-8-sig")
    print(f"saved partial US cases: {partial_path} | rows={len(partial_df)}")
    return partial_path


def save_us_partial_search_results(search_results: list[dict[str, Any]], suffix: str = "partial") -> Path:
    partial_path = OUTPUT_DIR / f"us_search_results_{suffix}.csv"
    partial_df = pd.DataFrame(search_results)
    partial_df.to_csv(partial_path, index=False, encoding="utf-8-sig")
    print(f"saved partial US search results: {partial_path} | rows={len(partial_df)}")
    return partial_path


def collect_us_cases_from_courtlistener_api() -> pd.DataFrame:
    validate_us_api_config(COURTLISTENER_API_TOKEN, COURTLISTENER_SEARCH_API_URL, COURTLISTENER_OPINIONS_API_URL)
    request_state = initialize_courtlistener_request_state()
    search_query = build_us_search_query(US_SEARCH_QUERY, US_EXTRA_QUERY)
    print(f"CourtListener query: {search_query}")
    search_stop_reason: Optional[str] = None
    try:
        search_results = fetch_courtlistener_search_results(
            search_url=COURTLISTENER_SEARCH_API_URL,
            api_token=COURTLISTENER_API_TOKEN,
            query=search_query,
            max_results=US_MAX_SEARCH_RESULTS,
            use_semantic_search=US_USE_SEMANTIC_SEARCH,
            request_state=request_state,
            extra_params=US_SEARCH_PARAMS,
        )
    except CourtListenerSearchStopped as exc:
        search_results = exc.collected_results
        search_stop_reason = str(exc)
        warnings.warn(f"US collection stopped before search completed: {exc}", UserWarning)
        save_us_partial_search_results(search_results, suffix="partial")
        if not search_results:
            save_us_partial_cases([], suffix="partial")
            return build_us_cases_dataframe([])
    except CourtListenerStop as exc:
        warnings.warn(f"US collection stopped before search completed: {exc}", UserWarning)
        save_us_partial_search_results([], suffix="partial")
        save_us_partial_cases([], suffix="partial")
        return build_us_cases_dataframe([])

    search_results_df = pd.DataFrame(search_results)
    display_preview(search_results_df, "US search results preview")
    if search_stop_reason:
        print(f"US search stopped early: {search_stop_reason}")

    rows: list[dict[str, Any]] = []
    skipped_missing_opinion = 0
    skipped_missing_text = 0
    stop_reason: Optional[str] = None

    for result in tqdm(search_results, total=len(search_results), desc="Fetch US opinion texts"):
        opinion_meta = select_preferred_opinion_meta(result.get("opinions"))
        if opinion_meta is None:
            skipped_missing_opinion += 1
            continue

        opinion_id = opinion_meta.get("id")
        try:
            opinion_payload = fetch_opinion_detail(COURTLISTENER_OPINIONS_API_URL, COURTLISTENER_API_TOKEN, opinion_id, request_state)
        except CourtListenerStop as exc:
            stop_reason = str(exc)
            warnings.warn(f"US collection stopped during opinion fetch: {exc}", UserWarning)
            save_us_partial_cases(rows, suffix="partial")
            break

        raw_text = extract_us_raw_text_from_opinion_payload(opinion_payload)
        if not raw_text:
            skipped_missing_text += 1
            continue

        citation_value = stringify_value(result.get("citation"))
        case_name = stringify_value(result.get("caseNameFull")) or stringify_value(result.get("caseName"))
        court_name = stringify_value(result.get("court")) or "unknown"
        suit_nature = stringify_value(result.get("suitNature"))
        case_id = f"CLUSTER_{result.get('cluster_id')}_OPINION_{opinion_id}"

        rows.append(
            {
                "case_id": case_id,
                "jurisdiction": "US",
                "country": "United States",
                "court_name": court_name,
                "court_level": infer_us_court_level(court_name),
                "decision_date": stringify_value(result.get("dateFiled")),
                "case_name": case_name,
                "case_number_or_citation": citation_value or stringify_value(result.get("docketNumber")),
                "source": "courtlistener_api",
                "case_type_keyword": suit_nature,
                "raw_text": raw_text,
                "clean_text": None,
                "reasoning_text": None,
                "token_count": 0,
                "sentence_count": 0,
            }
        )
        if len(rows) % US_PARTIAL_SAVE_EVERY == 0:
            save_us_partial_cases(rows, suffix="partial")

    us_df = build_us_cases_dataframe(rows)
    print(f"US collected cases: {len(us_df)}")
    print(f"US skipped results with no nested opinion id: {skipped_missing_opinion}")
    print(f"US skipped results with no opinion text: {skipped_missing_text}")
    print(f"US API requests made in this run: {request_state['made_this_run']}")
    if stop_reason:
        print(f"US collection stopped early: {stop_reason}")
    save_us_partial_cases(rows, suffix="latest")
    display_preview(us_df, "US collected cases")
    return us_df


In [3]:
validate_kr_dataset_config(KR_DATASET_NAME, KR_DATASET_CONFIG_NAME, KR_SPLIT, KR_COLUMN_MAP)
kr_raw_df = load_hf_dataset_as_dataframe(KR_DATASET_NAME, KR_DATASET_CONFIG_NAME, KR_SPLIT, "KR")

Loading KR dataset: lbox/lbox_open | config=precedent_corpus | split=train


[Kss]: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


KR columns (2): ['id', 'precedent']
KR raw dataset shape: (150000, 2)


,id,precedent
0,0,"주문\n상고를 모두 기각한다.\n상고비용은 원고들의 부담으로 한다.\n\n이유\n상고이유를 본다.\n원심판결 이유에 의하면 원심은, 대구칠곡지구 택지개발사업의 시행자인 피고는 그중 제2지구의 택지를 분양함에 있어서 대구직할시장으로부터 그 대금을 미리 받는 선수금의 승인을 받아, ..."
1,1,"주문\n상고를 기각한다.\n상고비용은 피고의 부담으로 한다.\n\n이유\n상고이유를 본다.\n1. 원심이 확정한 사실에 의하면, 소외 남양냉동식품주식회사(이하 소외회사라고 한다)가 1973. 8. 7. 서울 강남구 (주소 1 생략) 대 4,307.7m2를 취득하여 1984. 9...."
2,2,"주문\n원심판결을 파기하고, 사건을 서울고등법원에 환송한다.\n\n이유\n1. 피고의 상고이유를 판단한다.\n가. 종합토지세 과세표준액 경감대상토지의 면적산정방법에 관하여\n지방세법 제7조 제2항은 “지방자치단체는 공익상 기타의 사유로 인하여 필요한 때에는 불균일과세를 할 수 있..."


In [4]:
us_raw_df = collect_us_cases_from_courtlistener_api()

CourtListener query: ("damages" OR "civil damages" OR negligence OR tort OR "duty of care" OR "proximate cause" OR "comparative negligence" OR "compensatory damages" OR "punitive damages" OR "emotional distress" OR "personal injury" OR "breach of duty")
CourtListener search page 1: collected 20 results
CourtListener search page 2: collected 40 results
saved partial US cases: C:\Users\이정연\OneDrive\바탕 화면\comparative-law-llm\comparative-law-llm\outputs\us_cases_raw_partial.csv | rows=0


C:\Users\이정연\AppData\Local\Temp\ipykernel_19536\33992461.py:281: UserWarning: US collection stopped before search completed: CourtListener API rate limit reached.
  warnings.warn(f"US collection stopped before search completed: {exc}", UserWarning)


# 5. Normalize Dataset Schema

In [5]:
def warn_missing_mapped_columns(column_map: dict[str, Optional[str]], available_columns: set[str], label: str) -> None:
    for target_column, source_column in column_map.items():
        if source_column in (None, "TODO"):
            if target_column != "raw_text":
                warnings.warn(
                    f"[{label}] Mapping for '{target_column}' is None/TODO. Default values will be used.",
                    UserWarning,
                )
            continue
        if source_column not in available_columns:
            warnings.warn(
                f"[{label}] Mapped column '{source_column}' for '{target_column}' does not exist in dataset columns. Default values will be used.",
                UserWarning,
            )


def build_normalized_series(
    df: pd.DataFrame,
    source_column: Optional[str],
    default_value: Optional[str],
) -> pd.Series:
    if source_column in (None, "TODO") or source_column not in df.columns:
        return pd.Series([default_value] * len(df), index=df.index, dtype="object")
    return df[source_column].map(stringify_value)


def normalize_kr_dataset_schema(
    df: pd.DataFrame,
    column_map: dict[str, Optional[str]],
    dataset_name: str,
    config_name: Optional[str],
) -> pd.DataFrame:
    available_columns = set(df.columns)
    warn_missing_mapped_columns(column_map, available_columns, "KR")

    defaults = {
        "case_id": None,
        "court_name": "unknown",
        "court_level": "unknown",
        "decision_date": None,
        "case_name": None,
        "case_number_or_citation": None,
        "source": f"huggingface:{dataset_name}/{config_name}" if config_name else f"huggingface:{dataset_name}",
        "case_type_keyword": None,
        "raw_text": None,
    }

    normalized = pd.DataFrame(index=df.index)
    for target_column in [
        "case_id",
        "court_name",
        "court_level",
        "decision_date",
        "case_name",
        "case_number_or_citation",
        "source",
        "case_type_keyword",
        "raw_text",
    ]:
        normalized[target_column] = build_normalized_series(
            df=df,
            source_column=column_map.get(target_column),
            default_value=defaults[target_column],
        )

    normalized["jurisdiction"] = "KR"
    normalized["country"] = "Korea"

    missing_case_id_mask = normalized["case_id"].isna() | (normalized["case_id"].astype(str).str.strip() == "")
    if missing_case_id_mask.any():
        fallback_ids = [f"KR_{idx:07d}" for idx in normalized.index]
        normalized.loc[missing_case_id_mask, "case_id"] = pd.Series(fallback_ids, index=normalized.index)[missing_case_id_mask]

    normalized["source"] = normalized["source"].fillna(f"huggingface:{dataset_name}/{config_name}" if config_name else f"huggingface:{dataset_name}")
    normalized["court_name"] = normalized["court_name"].fillna("unknown")
    normalized["court_level"] = normalized["court_level"].fillna("unknown")
    normalized["raw_text"] = normalized["raw_text"].map(stringify_value)
    normalized["clean_text"] = None
    normalized["reasoning_text"] = None
    normalized["token_count"] = 0
    normalized["sentence_count"] = 0

    normalized = normalized[COMMON_SCHEMA_COLUMNS].copy()
    display_preview(normalized, "KR normalized")
    return normalized


def finalize_schema(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=COMMON_SCHEMA_COLUMNS)
    final_df = df.copy()
    for column in COMMON_SCHEMA_COLUMNS:
        if column not in final_df.columns:
            if column in {"token_count", "sentence_count"}:
                final_df[column] = 0
            else:
                final_df[column] = None
    final_df["token_count"] = pd.to_numeric(final_df["token_count"], errors="coerce").fillna(0).astype(int)
    final_df["sentence_count"] = pd.to_numeric(final_df["sentence_count"], errors="coerce").fillna(0).astype(int)
    return final_df[COMMON_SCHEMA_COLUMNS].copy()


kr_normalized_df = normalize_kr_dataset_schema(kr_raw_df, KR_COLUMN_MAP, KR_DATASET_NAME, KR_DATASET_CONFIG_NAME)
us_normalized_df = finalize_schema(us_raw_df)
display_preview(us_normalized_df, "US normalized")


C:\Users\이정연\AppData\Local\Temp\ipykernel_19536\1695831624.py:5: UserWarning: [KR] Mapping for 'court_name' is None/TODO. Default values will be used.
  warnings.warn(
C:\Users\이정연\AppData\Local\Temp\ipykernel_19536\1695831624.py:5: UserWarning: [KR] Mapping for 'court_level' is None/TODO. Default values will be used.
  warnings.warn(
C:\Users\이정연\AppData\Local\Temp\ipykernel_19536\1695831624.py:5: UserWarning: [KR] Mapping for 'decision_date' is None/TODO. Default values will be used.
  warnings.warn(
C:\Users\이정연\AppData\Local\Temp\ipykernel_19536\1695831624.py:5: UserWarning: [KR] Mapping for 'case_name' is None/TODO. Default values will be used.
  warnings.warn(
C:\Users\이정연\AppData\Local\Temp\ipykernel_19536\1695831624.py:5: UserWarning: [KR] Mapping for 'case_number_or_citation' is None/TODO. Default values will be used.
  warnings.warn(
C:\Users\이정연\AppData\Local\Temp\ipykernel_19536\1695831624.py:5: UserWarning: [KR] Mapping for 'source' is None/TODO. Default values will be use

KR normalized shape: (150000, 15)


,case_id,jurisdiction,country,court_name,court_level,decision_date,case_name,case_number_or_citation,source,case_type_keyword,raw_text,clean_text,reasoning_text,token_count,sentence_count
0,0,KR,Korea,unknown,unknown,None,None,None,huggingface:lbox/lbox_open/precedent_corpus,None,"주문\n상고를 모두 기각한다.\n상고비용은 원고들의 부담으로 한다.\n\n이유\n상고이유를 본다.\n원심판결 이유에 의하면 원심은, 대구칠곡지구 택지개발사업의 시행자인 피고는 그중 제2지구의 택지를 분양함에 있어서 대구직할시장으로부터 그 대금을 미리 받는 선수금의 승인을 받아, ...",None,None,0,0
1,1,KR,Korea,unknown,unknown,None,None,None,huggingface:lbox/lbox_open/precedent_corpus,None,"주문\n상고를 기각한다.\n상고비용은 피고의 부담으로 한다.\n\n이유\n상고이유를 본다.\n1. 원심이 확정한 사실에 의하면, 소외 남양냉동식품주식회사(이하 소외회사라고 한다)가 1973. 8. 7. 서울 강남구 (주소 1 생략) 대 4,307.7m2를 취득하여 1984. 9....",None,None,0,0
2,2,KR,Korea,unknown,unknown,None,None,None,huggingface:lbox/lbox_open/precedent_corpus,None,"주문\n원심판결을 파기하고, 사건을 서울고등법원에 환송한다.\n\n이유\n1. 피고의 상고이유를 판단한다.\n가. 종합토지세 과세표준액 경감대상토지의 면적산정방법에 관하여\n지방세법 제7조 제2항은 “지방자치단체는 공익상 기타의 사유로 인하여 필요한 때에는 불균일과세를 할 수 있...",None,None,0,0


US normalized shape: (0, 15)
US normalized is empty.


# 6. Text Cleaning and Preprocessing

In [6]:
TOKEN_PATTERN = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+|[가-힣]+")
MULTISPACE_PATTERN = re.compile(r"[ \t\x0b\x0c\r]+")
MULTINEWLINE_PATTERN = re.compile(r"\n{3,}")
MASKING_ARTIFACT_PATTERN = regex_module.compile(r"(?:[○●◇◆△▲□■▣◎◯]{2,}|[xX*＊]{2,}|_{2,})")
US_FOOTNOTE_LINE_PATTERN = re.compile(r"(?m)^\s*(?:\[\d+\]|\(\d+\)|\d+)\s*$")
US_INLINE_FOOTNOTE_PATTERN = re.compile(r"\[(\d{1,3})\]")
KR_REASONING_MARKER_PATTERN = re.compile(r"(?im)^\s*이\s*유\s*$")
KR_APPENDIX_PATTERN = re.compile(r"(?im)^\s*별\s*지\b")


def normalize_basic_text(text: Optional[str]) -> str:
    if text is None:
        return ""
    normalized = str(text).replace("\r\n", "\n").replace("\r", "\n")
    normalized = normalized.replace("\u00a0", " ").replace("\u200b", " ")
    normalized = MULTISPACE_PATTERN.sub(" ", normalized)
    normalized = re.sub(r"\n[ \t]+", "\n", normalized)
    normalized = MULTINEWLINE_PATTERN.sub("\n\n", normalized)
    return normalized.strip()


def remove_masking_artifacts(text: str) -> str:
    cleaned = MASKING_ARTIFACT_PATTERN.sub(" ", text)
    cleaned = MULTISPACE_PATTERN.sub(" ", cleaned)
    cleaned = MULTINEWLINE_PATTERN.sub("\n\n", cleaned)
    return cleaned.strip()


def extract_reasoning_text_kr(text: str) -> Optional[str]:
    match = KR_REASONING_MARKER_PATTERN.search(text)
    if not match:
        return None
    reasoning_text = text[match.end():].strip()
    return reasoning_text or None


def trim_appendix_kr(text: str, enabled: bool = True) -> str:
    if not enabled:
        return text
    match = KR_APPENDIX_PATTERN.search(text)
    if not match:
        return text
    return text[:match.start()].strip()


def clean_korean_case_text(raw_text: Optional[str], remove_appendix: bool = True) -> tuple[str, Optional[str]]:
    cleaned = normalize_basic_text(raw_text)
    cleaned = trim_appendix_kr(cleaned, enabled=remove_appendix)
    cleaned = remove_masking_artifacts(cleaned)
    reasoning_text = extract_reasoning_text_kr(cleaned)
    if reasoning_text is not None:
        reasoning_text = remove_masking_artifacts(reasoning_text)
    return cleaned, reasoning_text


def strip_us_front_matter(text: str) -> str:
    lowered = text.lower()
    cue_match = re.search(r"\b(opinion|memorandum opinion|opinion and order|discussion|analysis|background|facts)\b", lowered)
    if not cue_match:
        return text
    if cue_match.start() > 4000:
        return text
    prefix = lowered[:cue_match.start()]
    noise_signals = [
        "syllabus",
        "headnote",
        "headnotes",
        "attorney",
        "attorneys",
        "counsel",
        "for appellant",
        "for appellee",
        "table of authorities",
        "procedural history",
    ]
    if any(signal in prefix for signal in noise_signals):
        return text[cue_match.start():].strip()
    return text


def remove_us_noise_lines(text: str) -> str:
    cleaned_lines = []
    for line in text.split("\n"):
        stripped = line.strip()
        lowered = stripped.lower()
        if lowered in {"syllabus", "headnote", "headnotes", "table of authorities"}:
            continue
        if lowered.startswith("attorneys") or lowered.startswith("counsel"):
            continue
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines).strip()


def remove_us_footnote_noise(text: str) -> str:
    cleaned = US_FOOTNOTE_LINE_PATTERN.sub("", text)
    cleaned = US_INLINE_FOOTNOTE_PATTERN.sub("", cleaned)
    cleaned = re.sub(r"(?m)^\s*Page\s+\d+\s*$", "", cleaned)
    cleaned = MULTINEWLINE_PATTERN.sub("\n\n", cleaned)
    return cleaned.strip()


def clean_us_case_text(raw_text: Optional[str]) -> str:
    cleaned = normalize_basic_text(raw_text)
    cleaned = strip_us_front_matter(cleaned)
    cleaned = remove_us_noise_lines(cleaned)
    cleaned = remove_us_footnote_noise(cleaned)
    cleaned = remove_masking_artifacts(cleaned)
    return cleaned


def split_sentences_kr(text: str) -> list[str]:
    if not text.strip():
        return []
    if KSS_AVAILABLE:
        try:
            return [sentence.strip() for sentence in kss.split_sentences(text) if sentence.strip()]
        except Exception as exc:
            print(f"[warning] kss sentence split failed. Falling back to regex. reason={exc}")
    sentences = re.split(r"(?<=[.!?]|다\.|요\.|죠\.)\s+|\n+", text)
    return [sentence.strip() for sentence in sentences if sentence.strip()]


def split_sentences_us(text: str) -> list[str]:
    if not text.strip():
        return []
    try:
        from nltk.tokenize import sent_tokenize
        return [sentence.strip() for sentence in sent_tokenize(text) if sentence.strip()]
    except Exception as exc:
        print(f"[warning] nltk sent_tokenize failed. Falling back to regex. reason={exc}")
    sentences = re.split(r"(?<=[.!?])\s+|\n+", text)
    return [sentence.strip() for sentence in sentences if sentence.strip()]


def count_tokens(text: Optional[str]) -> int:
    if text is None:
        return 0
    normalized = normalize_basic_text(text)
    if not normalized:
        return 0
    tokens = TOKEN_PATTERN.findall(normalized)
    if tokens:
        return len(tokens)
    return len(normalized.split())


def count_sentences(text: Optional[str], jurisdiction: str) -> int:
    if text is None:
        return 0
    sentences = split_sentences_kr(text) if jurisdiction == "KR" else split_sentences_us(text)
    return len(sentences)


def preprocess_cases(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=COMMON_SCHEMA_COLUMNS)

    processed_rows = []
    jurisdiction_label = stringify_value(df["jurisdiction"].iloc[0]) or "unknown"
    for row in tqdm(df.to_dict(orient="records"), total=len(df), desc=f"Preprocess {jurisdiction_label}"):
        raw_text = stringify_value(row.get("raw_text"))
        jurisdiction = row["jurisdiction"]

        if jurisdiction == "KR":
            clean_text, reasoning_text = clean_korean_case_text(raw_text, remove_appendix=REMOVE_KR_APPENDIX_AFTER_BYEOLJI)
        else:
            clean_text = clean_us_case_text(raw_text)
            reasoning_text = None

        row["raw_text"] = raw_text
        row["clean_text"] = clean_text or None
        row["reasoning_text"] = reasoning_text
        row["token_count"] = count_tokens(clean_text)
        row["sentence_count"] = count_sentences(clean_text, jurisdiction=jurisdiction)
        processed_rows.append(row)

    processed_df = pd.DataFrame(processed_rows)
    processed_df = finalize_schema(processed_df)
    print(f"{jurisdiction_label} preprocessed shape: {processed_df.shape}")
    display(processed_df[["case_id", "case_name", "token_count", "sentence_count", "clean_text", "reasoning_text"]].head(PREVIEW_ROWS))
    return processed_df


In [ ]:
kr_preprocessed_df = preprocess_cases(kr_normalized_df)
us_preprocessed_df = preprocess_cases(us_normalized_df)

short_kr_count = int((kr_preprocessed_df["token_count"] < MIN_TOKENS).sum()) if not kr_preprocessed_df.empty else 0
short_us_count = int((us_preprocessed_df["token_count"] < MIN_TOKENS).sum()) if not us_preprocessed_df.empty else 0
print(f"KR cases below MIN_TOKENS before filtering: {short_kr_count}")
print(f"US cases below MIN_TOKENS before filtering: {short_us_count}")


Preprocess KR:   0%|          | 0/150000 [00:00<?, ?it/s]

[Kss]: Because there's no supported C++ morpheme analyzer, Kss will take pecab as a backend. :D
For your information, Kss also supports mecab backend.
We recommend you to install mecab or konlpy.tag.Mecab for faster execution of Kss.
Please refer to following web sites for details:
- mecab: https://cleancode-ws.tistory.com/97
- konlpy.tag.Mecab: https://uwgdqo.tistory.com/363

c:\Users\이정연\OneDrive\바탕 화면\comparative-law-llm\comparative-law-llm\.venv\Lib\site-packages\pecab\_tokenizer.py:265: RuntimeWarning: overflow encountered in scalar add
  from_pos_data.costs[idx]
c:\Users\이정연\OneDrive\바탕 화면\comparative-law-llm\comparative-law-llm\.venv\Lib\site-packages\pecab\_tokenizer.py:274: RuntimeWarning: overflow encountered in scalar add
  least_cost += word_cost


# 7. Filter Damages/Tort-related Cases

In [ ]:
def find_matched_keywords(text: Optional[str], keywords: list[str], ignore_case: bool) -> list[str]:
    normalized = stringify_value(text) or ""
    if ignore_case:
        lowered = normalized.lower()
        return [keyword for keyword in keywords if keyword.lower() in lowered]
    return [keyword for keyword in keywords if keyword in normalized]


def filter_related_cases(df: pd.DataFrame, keywords: list[str], jurisdiction: str) -> pd.DataFrame:
    if df.empty:
        warnings.warn(f"[{jurisdiction}] Input dataframe is empty before keyword filtering.", UserWarning)
        return df.copy()

    ignore_case = jurisdiction == "US"
    working_df = df.copy()

    def _row_matches(row: pd.Series) -> list[str]:
        search_text = "\n".join(
            [
                stringify_value(row.get("raw_text")) or "",
                stringify_value(row.get("case_type_keyword")) or "",
            ]
        )
        return find_matched_keywords(search_text, keywords=keywords, ignore_case=ignore_case)

    working_df["_matched_keywords"] = working_df.progress_apply(_row_matches, axis=1)
    before_count = len(working_df)
    filtered_df = working_df[working_df["_matched_keywords"].map(bool)].copy()
    after_count = len(filtered_df)
    print(f"{jurisdiction} keyword filter: {before_count} -> {after_count}")
    if after_count == 0:
        warnings.warn(f"[{jurisdiction}] No rows remained after keyword filtering.", UserWarning)
    display(filtered_df[["case_id", "case_name", "case_type_keyword", "token_count", "_matched_keywords"]].head(PREVIEW_ROWS))
    return filtered_df


def filter_by_min_tokens(df: pd.DataFrame, min_tokens: int, jurisdiction: str) -> pd.DataFrame:
    if df.empty:
        warnings.warn(f"[{jurisdiction}] Input dataframe is empty before min token filtering.", UserWarning)
        return df.copy()

    before_count = len(df)
    filtered_df = df[df["token_count"] >= min_tokens].copy()
    removed_count = before_count - len(filtered_df)
    print(f"{jurisdiction} min token filter (>= {min_tokens}): {before_count} -> {len(filtered_df)} | removed={removed_count}")
    if len(filtered_df) == 0:
        warnings.warn(f"[{jurisdiction}] No rows remained after min token filtering.", UserWarning)
    return filtered_df


kr_keyword_filtered_df = filter_related_cases(kr_preprocessed_df, KR_KEYWORDS, "KR")
us_keyword_filtered_df = filter_related_cases(us_preprocessed_df, US_KEYWORDS, "US")

kr_filtered_df = filter_by_min_tokens(kr_keyword_filtered_df, MIN_TOKENS, "KR")
us_filtered_df = filter_by_min_tokens(us_keyword_filtered_df, MIN_TOKENS, "US")


# 8. Sample Pilot Corpus

In [ ]:
def sample_pilot_corpus(df: pd.DataFrame, sample_size: int, jurisdiction: str, seed: int) -> pd.DataFrame:
    available_count = len(df)
    if available_count < sample_size:
        warnings.warn(
            f"[{jurisdiction}] Requested sample_size={sample_size}, but only {available_count} cases are available. Using all available rows.",
            UserWarning,
        )
    sample_n = min(sample_size, available_count)
    if sample_n == 0:
        return finalize_schema(df.head(0).copy())
    sampled_df = df.sample(n=sample_n, random_state=seed).reset_index(drop=True)
    print(f"{jurisdiction} sampled count: {len(sampled_df)}")
    display(sampled_df[["case_id", "case_name", "court_name", "court_level", "decision_date", "token_count"]].head(PREVIEW_ROWS))
    return finalize_schema(sampled_df)


kr_sampled_df = sample_pilot_corpus(kr_filtered_df, SAMPLE_SIZE_PER_JURISDICTION, "KR", SEED)
us_sampled_df = sample_pilot_corpus(us_filtered_df, SAMPLE_SIZE_PER_JURISDICTION, "US", SEED)

print("Final sampled counts by jurisdiction:")
print(kr_sampled_df["jurisdiction"].value_counts(dropna=False) if not kr_sampled_df.empty else "KR empty")
print(us_sampled_df["jurisdiction"].value_counts(dropna=False) if not us_sampled_df.empty else "US empty")


# 9. Save Cleaned Corpus

In [ ]:
def save_cleaned_csv(df: pd.DataFrame, path: Path) -> None:
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"saved: {path}")


kr_final_df = finalize_schema(kr_sampled_df)
us_final_df = finalize_schema(us_sampled_df)
combined_final_df = finalize_schema(pd.concat([kr_final_df, us_final_df], ignore_index=True))

kr_output_path = OUTPUT_DIR / "kr_cases_cleaned.csv"
us_output_path = OUTPUT_DIR / "us_cases_cleaned.csv"
combined_output_path = OUTPUT_DIR / "combined_cases_cleaned.csv"

save_cleaned_csv(kr_final_df, kr_output_path)
save_cleaned_csv(us_final_df, us_output_path)
save_cleaned_csv(combined_final_df, combined_output_path)


# 10. Quick Sanity Checks

In [ ]:
def sanity_check_summary(df: pd.DataFrame, label: str) -> None:
    print(f"\n[{label}] shape: {df.shape}")
    print(f"[{label}] columns: {list(df.columns)}")
    if df.empty:
        print(f"[{label}] dataframe is empty.")
        return
    print(f"[{label}] jurisdiction counts:")
    print(df["jurisdiction"].value_counts(dropna=False))
    print(f"[{label}] empty raw_text count: {int(df['raw_text'].isna().sum())}")
    print(f"[{label}] token_count == 0 count: {int((df['token_count'] == 0).sum())}")
    print(f"[{label}] token_count summary:")
    print(df["token_count"].describe())
    print(f"[{label}] sentence_count summary:")
    print(df["sentence_count"].describe())
    display(df.head(PREVIEW_ROWS))


sanity_check_summary(kr_final_df, "KR final")
sanity_check_summary(us_final_df, "US final")
sanity_check_summary(combined_final_df, "Combined final")

print("\nSaved output files:")
print(kr_output_path)
print(us_output_path)
print(combined_output_path)
